In [1]:
import os
import json
import chromadb
from collections import defaultdict

In [2]:

from groq import Groq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [3]:
MODEL_NAME = "openai/gpt-oss-120b"
COLLECTION_NAME = "tesla-10k-2019-to-2023"

OUTPUT_DIR = "./outputs"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "rag_results.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
GROQ_API_KEY =os.getenv("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

C:\Users\Simran Kumar\AppData\Local\Temp\ipykernel_2652\2252124784.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
c:\Users\Simran Kumar\Documents\query expansion_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
chromadb_client = chromadb.PersistentClient(path="./tesla_db")

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [7]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

PROMPTS

In [8]:
EXPANSION_PROMPT = """
Generate 4–6 different search queries that preserve meaning of the question.
Return only newline separated queries.
"""

QNA_SYSTEM = """
You are a financial RAG assistant.
Answer ONLY using provided context.
If not found, say "I don't know".
"""

QNA_TEMPLATE = """
<Context>
{context}
</Context>

<Question>
{question}
</Question>
"""

In [9]:
def call_llm(prompt, system=QNA_SYSTEM):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()

QUERY EXPANSION

In [10]:
def expand_query(query):
    result = call_llm(
        f"{EXPANSION_PROMPT}\n\nQuestion: {query}",
        system=EXPANSION_PROMPT
    )

    return [q.strip() for q in result.split("\n") if q.strip()]

EXPANDED RETRIEVAL

In [11]:
def expanded_retrieval(queries):

    score_map = defaultdict(float)
    doc_store = {}

    for q in queries:
        docs = retriever.invoke(q)

        for rank, doc in enumerate(docs):
            text = doc.page_content

            score_map[text] += 1 / (rank + 1)
            doc_store[text] = text

    ranked = sorted(score_map.items(), key=lambda x: x[1], reverse=True)

    return [doc_store[t] for t, _ in ranked]

ANSWER

In [12]:
def generate_answer(query, docs):

    context = "\n\n".join(docs)

    prompt = QNA_TEMPLATE.format(
        context=context,
        question=query
    )

    return call_llm(prompt)

In [13]:
def run_pipeline(question_id, query):

    expanded_queries = expand_query(query)

    expanded_docs = expanded_retrieval(expanded_queries)

    final_answer = generate_answer(query, expanded_docs)

    return {
        "question_id": question_id,
        "original_query": query,
        "expanded_queries": expanded_queries,
        "final_answer": final_answer
    }

BENCHMARK QUESTIONS

In [14]:
benchmark_questions = {
    "Q1": "Does Tesla's growth story appear more constrained by external supply risk or internal execution?",
    "Q2": "How is Tesla AI reflected in spending and strategy?",
    "Q3": "Is Tesla exposed to supplier or geographic concentration risk?",
    "Q4": "Compare automotive vs energy business importance"
}

JSON

In [15]:
results = []

for qid, query in benchmark_questions.items():
    results.append(run_pipeline(qid, query))

final_output = {
    "results": results
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=4)

print(f"Saved clean output at: {OUTPUT_FILE}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Saved clean output at: ./outputs\rag_results.json
